# Packages import

In [37]:
import os
import yaml
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

# Apollo Scraper

In [38]:
with open("config.yaml", "r", encoding="UTF-8") as yf:
    config = yaml.safe_load(yf)
username = config['credentials']['user']
password = config['credentials']['password']
credentials = HTTPBasicAuth(username, password)

In [39]:

url = "https://planzajec.uek.krakow.pl/index.php?typ=G&id=252681&okres=2"
response = requests.get(url, auth=credentials)
response.encoding = "UTF-8"
print(response.status_code)

200


In [40]:
page_dom = BeautifulSoup(response.text, 'html.parser')

In [41]:
group = page_dom.select_one("div.grupa").get_text(strip=True)
print(group)

ZICSS1-1211


In [42]:
classes_tag = page_dom.select_one("table")
with open("temp.html","w",encoding="UTF-8") as hf:
    hf.write(classes_tag.prettify())
classes = pd.read_html("temp.html", encoding="UTF-8")[0]
os.remove("temp.html")

In [43]:
classes = classes.loc[classes["Typ"].isin(["Wykład", "ćwiczenia", "egzamin"])]

In [44]:
classes[['Day','Start Time','hyphen', 'End Time', 'Duration']] = classes['Dzień, Godzina'].str.split(' ', expand=True)

KeyError: 'Dzień, Godzina'

In [ ]:
classes['Duration'] = classes['Duration'].map(lambda x: x.split('(')[1].split('g')[0])

In [ ]:
classes = classes.drop(['Dzień, Godzina','hyphen'], axis=1)

In [ ]:
classes['Sala'] = classes['Sala'].str.replace(
    r"(lab\.),*"
    r'\1',
    regex=True
)

In [ ]:
if not os.path.exists("schedules"):
    os.mkdir("schedules")

In [ ]:
classes.to_csv(f"schedules/{group}.csv")